# Evaluation

Pipeline step after training. Loads a trained checkpoint and evaluates it - no training, so a crashed or finished run can be evaluated at any time, and every run in the grid goes through identical code.

Extraction and matching live in `crater_extraction.py`, shared with the baseline.

Comparable to DeepMoon's *post-CNN* column (57% recall), not their *post-processed* 92% - that figure comes from merging detections across ~120 overlapping views per crater, which this single-patch setup does not do.

In [ ]:
import sys
sys.path.append('../data_extraction')
sys.path.append('../training')

import os
import numpy as np
import pandas as pd
import mlflow
import keras
import matplotlib.pyplot as plt

from crater_extraction import template_match_t, match_coords, filter_to_detectable, filter_edge_craters, truth_coords_for_patch
from LRO_data_class import getSplitIndices, percentileNormalise

In [ ]:

# the evaluated run (testing purposes): builds the checkpoint and history filenames
DATASET = 'single'

MODEL = 'U-Net-v1'
CHANNELS = 'both'
N_FILTERS = 32
SEED = 42
EPOCH = 5

if DATASET == 'single':
    PATCHES_DIR = '../pre_processing/lunar_patches'
    LABELS_CSV = '../data_preparation/filtered_labels.csv'
else:
    PATCHES_DIR = '../pre_processing/lunar_patches_alltiles'
    LABELS_CSV = '../data_preparation/filtered_labels_alltiles.csv'

CHECKPOINT_DIR = '../training/checkpoints'

CHECKPOINT = os.path.join(CHECKPOINT_DIR, f'{MODEL}_{CHANNELS}_{N_FILTERS}f_s{SEED}_{EPOCH:02d}.keras')
HISTORY_CSV = os.path.join(CHECKPOINT_DIR, f'history_{MODEL}_{CHANNELS}_{N_FILTERS}f_s{SEED}.csv')

kept_labels = pd.read_csv(os.path.join(PATCHES_DIR, 'kept_labels.csv'), low_memory=False)
filtered_labels = pd.read_csv(LABELS_CSV)

train_idx, val_idx, test_idx = getSplitIndices(PATCHES_DIR)

print(f'train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}')

In [ ]:
model = keras.models.load_model(CHECKPOINT)

mlflow.set_tracking_uri('../training/mlruns')
mlflow.set_experiment('lunar-crater-detection')

with mlflow.start_run(run_name=f'eval-{CHANNELS}-{N_FILTERS}f-s{SEED}') as run:
    mlflow.log_param('checkpoint', CHECKPOINT)
    mlflow.log_param('dataset', DATASET)
    mlflow.log_param('channels', CHANNELS)
    mlflow.log_param('n_filters', N_FILTERS)
    mlflow.log_param('seed', SEED)

    run_id = run.info.run_id

print(CHECKPOINT)

In [ ]:
# filtered_labels.csv holds only Robbins columns - wac_col/wac_row are added in
# data_pre_processing after the csv is saved, so they survive only in kept_labels.
# every tile measures pixels from its own top-left corner, so the fit is per tile.
# craters outside a tile map beyond its raster and the patch window drops them.
if 'tile' not in kept_labels.columns:
    kept_labels['tile'] = 'single'

tile_craters = {}

for tile_name in kept_labels['tile'].dropna().unique():

    tile_rows = kept_labels[kept_labels['tile'] == tile_name]
    crater_rows = tile_rows.dropna(subset=['LON_CIRC_IMG', 'wac_col'])

    col_fit = np.polyfit(crater_rows['LON_CIRC_IMG'], crater_rows['wac_col'], 1)
    row_fit = np.polyfit(crater_rows['LAT_CIRC_IMG'], crater_rows['wac_row'], 1)

    tile_wac_col = np.polyval(col_fit, filtered_labels['LON_CIRC_IMG'].values)
    tile_wac_row = np.polyval(row_fit, filtered_labels['LAT_CIRC_IMG'].values)

    tile_craters[tile_name] = (tile_wac_col, tile_wac_row, filtered_labels['DIAM_CIRC_IMG'].values)

    print(f'{tile_name}: lon -> col {col_fit[0]:.2f} px/deg     lat -> row {row_fit[0]:.2f} px/deg')

In [ ]:
# patches are stored 1000 per file, so the file is held between calls and only
# reloaded when the index crosses into the next one
loaded = {}


def patchInput(patch_idx):

    file_num = int(patch_idx // 1000)
    position = patch_idx % 1000

    if loaded.get('file') != file_num:
        loaded['wac'] = np.load(os.path.join(PATCHES_DIR, f'X_wac_{file_num}.npz'))['arr_0']
        loaded['dem'] = np.load(os.path.join(PATCHES_DIR, f'X_dem_{file_num}.npz'))['arr_0']
        loaded['file'] = file_num

    wac_patch = percentileNormalise(loaded['wac'][position])
    dem_patch = percentileNormalise(loaded['dem'][position])

    if CHANNELS == 'both':
        return np.stack([wac_patch, dem_patch], axis=-1)

    if CHANNELS == 'wac':
        return wac_patch[..., None]

    return dem_patch[..., None]


def patchTruth(patch_idx):

    row = kept_labels.iloc[patch_idx]
    tile_wac_col, tile_wac_row, tile_diameters = tile_craters[row['tile']]

    truth = truth_coords_for_patch(row['center_col'], row['center_row'], row['patch_lat'],
                                   tile_wac_col, tile_wac_row, tile_diameters)

    return filter_edge_craters(filter_to_detectable(truth))

## Threshold sweep

In [ ]:
# target_thresh inherited from DeepMoon at 0.1, where the loss was unweighted BCE.
# focal squashes the output into a narrow low band, so 0.1 can binarise the whole
# patch into one blob and match nothing. both tails go to zero - notes 17.3
# runs on VALIDATION - tuning a threshold on test contaminates every number after it

thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7]

n_sweep = 200

sweep_rng = np.random.default_rng(SEED)
sweep_idx = np.sort(sweep_rng.choice(val_idx, size=n_sweep, replace=False))

# predictions and truth are the same at every threshold, so build them once
sweep_pred = []
sweep_truth = []

for patch_idx in sweep_idx:

    sweep_pred.append(model.predict(patchInput(patch_idx)[None, ...], verbose=0)[0, :, :, 0])
    sweep_truth.append(patchTruth(patch_idx))

sweep_precision = []
sweep_recall = []
sweep_f1 = []

for threshold in thresholds:

    swept_match = 0
    swept_detected = 0
    swept_truth = 0

    for prediction, truth in zip(sweep_pred, sweep_truth):

        detections = filter_edge_craters(template_match_t(prediction.copy(), target_thresh=threshold))

        match_count, detection_count, truth_count, _, _, _ = match_coords(truth, detections)

        swept_match += match_count
        swept_detected += detection_count
        swept_truth += truth_count

    if swept_detected > 0:
        sweep_precision.append(swept_match / swept_detected)
    else:
        sweep_precision.append(0)

    if swept_truth > 0:
        sweep_recall.append(swept_match / swept_truth)
    else:
        sweep_recall.append(0)

    if sweep_precision[-1] + sweep_recall[-1] > 0:
        sweep_f1.append(2 * sweep_precision[-1] * sweep_recall[-1] / (sweep_precision[-1] + sweep_recall[-1]))
    else:
        sweep_f1.append(0)

    print(f'threshold {threshold}: P {sweep_precision[-1]:.3f}  R {sweep_recall[-1]:.3f}  F1 {sweep_f1[-1]:.3f}')

# the value carried into evaluation. the optimum is interior, so it is read off here
# rather than assumed - a flat zero row means the grid missed it, widen and re-run
best_threshold = thresholds[int(np.argmax(sweep_f1))]

print(f'best threshold: {best_threshold}   F1 {max(sweep_f1):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(thresholds, sweep_precision, marker='o', label='precision')
axes[0].plot(thresholds, sweep_recall, marker='o', label='recall')
axes[0].plot(thresholds, sweep_f1, marker='o', label='F1')
axes[0].set_xlabel('target_thresh')
axes[0].legend()

axes[1].plot(sweep_recall, sweep_precision, marker='o')
axes[1].set_xlabel('recall')
axes[1].set_ylabel('precision')

plt.show()

## Crater-level metrics

In [ ]:
# iterates test_idx directly, not a generator - the generator shuffles and never
# says which patch it handed back, so its patches cannot be looked up in kept_labels

n_eval = 160

rng = np.random.default_rng(SEED)
# sorted so patches from the same npz are consecutive and the file can be reused
eval_idx = np.sort(rng.choice(test_idx, size=n_eval, replace=False))

total_match = 0
total_detected = 0
total_truth = 0
total_multi_match = 0

all_matched_pairs = []
all_false_positives = []
all_truth_radii = []

for patch_idx in eval_idx:

    prediction = model.predict(patchInput(patch_idx)[None, ...], verbose=0)

    detections = filter_edge_craters(template_match_t(prediction[0, :, :, 0].copy(), target_thresh=best_threshold))

    truth = patchTruth(patch_idx)

    match_count, detection_count, truth_count, matched_pairs, false_positives, multi_match_count = match_coords(truth, detections)

    total_match += match_count
    total_detected += detection_count
    total_truth += truth_count
    total_multi_match += multi_match_count

    if len(matched_pairs) > 0:
        all_matched_pairs.append(matched_pairs)

    if len(false_positives) > 0:
        all_false_positives.append(false_positives)

    if len(truth) > 0:
        all_truth_radii.append(truth[:, 2])

if len(all_matched_pairs) > 0:
    all_matched_pairs = np.vstack(all_matched_pairs)
else:
    all_matched_pairs = np.empty((0, 6))

if len(all_false_positives) > 0:
    all_false_positives = np.vstack(all_false_positives)
else:
    all_false_positives = np.empty((0, 3))

if len(all_truth_radii) > 0:
    all_truth_radii = np.concatenate(all_truth_radii)
else:
    all_truth_radii = np.empty(0)

print(f'TP: {total_match}   detected: {total_detected}   truth: {total_truth}')
print(f'detections claiming >1 truth crater: {total_multi_match}')

In [ ]:
if total_detected > 0:
    precision = total_match / total_detected
else:
    precision = 0

if total_truth > 0:
    recall = total_match / total_truth
else:
    recall = 0

if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0

print(f'P: {precision:.3f}   R: {recall:.3f}   F1: {f1:.3f}')

with mlflow.start_run(run_id=run_id):
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('n_eval_patches', n_eval)
    mlflow.log_metric('multi_match', total_multi_match)
    mlflow.log_metric('target_thresh', best_threshold)

## Figures

In [ ]:
# Loss curves
# read from the CSVLogger file, not a history object - so this works for a loaded
# checkpoint and survives a crashed run, since it is written each epoch

curves = pd.read_csv(HISTORY_CSV)

best_epoch = int(curves['val_loss'].idxmin())

plt.plot(curves['loss'], label='train')
plt.plot(curves['val_loss'], label='val')
plt.axvline(best_epoch, color='grey', linestyle='--', label=f'best epoch ({best_epoch + 1})')

plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()

In [ ]:
# Recall by crater diameter
# diameter_km = 2 * r_px * 0.1 -> bins 1-2 / 2-5 / 5-10 km are r = 5 / 10 / 25 / 50
# DeepMoon: recall drops above r = 15 px, 3 km here. notes 13.1

bin_edges = [5, 10, 25, 50]
bin_labels = ['1-2 km', '2-5 km', '5-10 km']

matched_truth_radii = all_matched_pairs[:, 5]

bin_recall = []

for lower, upper in zip(bin_edges[:-1], bin_edges[1:]):

    truth_in_bin = ((all_truth_radii >= lower) & (all_truth_radii < upper)).sum()
    matched_in_bin = ((matched_truth_radii >= lower) & (matched_truth_radii < upper)).sum()

    if truth_in_bin > 0:
        bin_recall.append(matched_in_bin / truth_in_bin)
    else:
        bin_recall.append(0)

    print(f'{lower}-{upper} px: {matched_in_bin} / {truth_in_bin}')

plt.bar(bin_labels, bin_recall)
plt.ylabel('recall')
plt.ylim(0, 1)
plt.title('Recall by crater diameter')
plt.show()

In [ ]:
# Crater size-frequency distribution
# parallel to the catalogue -> extra detections behave like real craters

all_detected_radii = np.concatenate([all_matched_pairs[:, 2], all_false_positives[:, 2]])

detected_diameters = all_detected_radii * 2 * 0.1
truth_diameters = all_truth_radii * 2 * 0.1

diameter_bins = np.logspace(np.log10(1), np.log10(10), 15)

plt.hist(truth_diameters, bins=diameter_bins, histtype='step', label='Robbins (in patch)')
plt.hist(detected_diameters, bins=diameter_bins, histtype='step', label='detected')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('diameter (km)')
plt.ylabel('count')
plt.legend()
plt.show()

In [ ]:
# Positional and radius error
# fractional, over the mean radius. DeepMoon medians <= 11% (Table 3.1)

mean_radius = (all_matched_pairs[:, 2] + all_matched_pairs[:, 5]) / 2

error_x = abs(all_matched_pairs[:, 0] - all_matched_pairs[:, 3]) / mean_radius
error_y = abs(all_matched_pairs[:, 1] - all_matched_pairs[:, 4]) / mean_radius
error_radius = abs(all_matched_pairs[:, 2] - all_matched_pairs[:, 5]) / mean_radius

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, values, name in zip(axes, [error_x, error_y, error_radius], ['x', 'y', 'radius']):

    ax.hist(values, bins=40)
    ax.axvline(np.median(values), color='red', linestyle='--')
    ax.set_title(f'{name}: median {np.median(values):.3f}')
    ax.set_xlabel('fractional error')

plt.show()

In [ ]:
# False positives
# Robbins incomplete near 1 km - some of these are real, uncatalogued.
# large-radius ones sitting on obvious craters are the >= 10 km catalogue cut,
# not model error - the imagery was never filtered, only the labels

n_show = 12

n_show = min(n_show, len(all_false_positives))
sample = all_false_positives[rng.choice(len(all_false_positives), n_show, replace=False)]

print('false positives: ', len(all_false_positives))
print(sample)

## Alignment check

Three independently derived things on one image: the WAC patch, the stored mask, and the Robbins truth. Rings on visible craters in both panels, and the panels agreeing, means the coordinate chain is sound.

In [ ]:
patch_idx = test_idx[0]

file_num = int(patch_idx // 1000)
position = patch_idx % 1000

raw_wac = np.load(os.path.join(PATCHES_DIR, f'X_wac_{file_num}.npz'))['arr_0']
raw_mask = np.load(os.path.join(PATCHES_DIR, f'X_mask_{file_num}.npz'))['arr_0']

wac_patch = percentileNormalise(raw_wac[position])
mask_patch = raw_mask[position]

truth = patchTruth(patch_idx)

fig, ax = plt.subplots(1, 2, figsize=(13, 6))

ax[0].imshow(wac_patch, cmap='gray')
ax[0].imshow(np.ma.masked_where(mask_patch == 0, mask_patch), cmap='autumn')
ax[0].set_title('WAC + stored mask')

ax[1].imshow(wac_patch, cmap='gray')

for x, yy, r in truth:
    ax[1].add_patch(plt.Circle((x, yy), r, fill=False, color='red'))

ax[1].set_title(f'WAC + Robbins ({len(truth)})')

plt.show()